# Welcome to **futurezim notebooks**

You have a private JupyterLab server with a dedicated slice of an NVIDIA H200.
Run the cells below to see exactly what you've been given — everything here is live.

---


## 1. Your GPU

You get one **MIG slice**: a hardware-isolated partition of the H200 with **18 GB of VRAM**.
It is yours alone while your server runs — no one else's job can slow it down or run you out of memory.


In [ ]:
import torch

print('GPU available :', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('device        :', p.name)
    print('VRAM          : %.1f GB' % (p.total_memory / 1e9))
else:
    print('You are on the CPU-only profile. Restart from the Hub control panel')
    print('to pick a GPU profile: File > Hub Control Panel > Stop, then Start.')


## 2. Where to put things

This is the part worth reading. **Only your home directory is private and backed up.**

| Path | What it is | Shared? | Survives server restart? |
|---|---|---|---|
| `~/` (home) | Your work. Quota-enforced. | Private | Yes — backed up nightly |
| `~/shared/` | Curated datasets, read-only | Everyone reads | Yes |
| `~/team/` | Team scratch, read-write | Everyone reads *and writes* | Yes |
| `~/spaces` (object storage) | Big files, archives | Private to you | Yes |
| `/tmp` | Scratch | No | **No — wiped on restart** |

Put datasets everyone needs in `~/team/` rather than each keeping a copy — your
home quota is modest on purpose, and duplicated data is what fills the disk.


In [ ]:
import shutil, os

for label, path in [('home', os.path.expanduser('~')),
                    ('shared (ro)', os.path.expanduser('~/shared')),
                    ('team (rw)', os.path.expanduser('~/team'))]:
    if os.path.exists(path):
        t, u, f = shutil.disk_usage(path)
        print(f'{label:14} {u/2**30:6.1f} GB used  /  {t/2**30:6.1f} GB total')
    else:
        print(f'{label:14} not mounted')


## 3. Object storage — for anything large

Your home quota is sized for code and working data, not for archives or checkpoints.
Everything bigger belongs in object storage, which is off-box and effectively unlimited.

This is the equivalent of mounting Drive in Colab, but as a command rather than a mount:

```bash
spaces ls              # what's in your space
spaces put model.pt    # upload
spaces get model.pt    # download
spaces du              # how much you're storing
```

It also works from Python — `boto3` is preconfigured, no credentials needed:


In [ ]:
import os

if os.environ.get('S3_ENDPOINT'):
    import boto3
    s3 = boto3.client('s3', endpoint_url=os.environ['S3_ENDPOINT'])
    prefix = f"users/{os.environ['JUPYTERHUB_USER']}/"
    r = s3.list_objects_v2(Bucket=os.environ['S3_BUCKET'], Prefix=prefix)
    print('objects in your space:', r.get('KeyCount', 0))
else:
    print('Object storage is not configured on this deployment yet.')


## 4. AI assistance

The `caimex` CLI is preinstalled and already signed in — the equivalent of Colab's
built-in assistant, but it works on your whole project, not just one cell.
Open a Terminal (**File → New → Terminal**) and run:

```bash
caimex          # interactive session in the current directory
```


## 5. Your server shuts down when idle

There are only **7 GPU slices** on this machine. An idle notebook holding one is a
slice nobody else can use, so servers are stopped automatically after **1 hour idle**.

**Your files are never affected** — home, `team/` and object storage all persist.
Only the running process stops. Start again from the hub and pick up where you left off.

If you need a long unattended run, keep the notebook actively executing; a running
cell counts as activity.


## 6. Getting code in and out

- **Git** — the folder icon in the left sidebar has a Git tab: clone, commit, push.
- **Open something from GitHub** — use an `nbgitpuller` link, which clones a repo
  into your home directory and opens it, merging cleanly if you already have it.
- **Upload / download** — drag files into the file browser, or right-click → Download.

## 7. Keeping an eye on resources

The **status bar at the bottom** shows live memory and CPU. For the GPU, open
**NVDashboard** from the left sidebar to watch utilisation and VRAM while you work.

---

Questions, or need a bigger quota? Contact your administrator.

*You can delete this notebook — it won't come back.*
